# 03 Konvolusjonsnettverk (CNN)
*Data Mining & Analytics – Samling 3, dag 1*

Pensum: Géron kap. 13 (s. 353–376).

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
rng = np.random.default_rng(42)
tf.keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__)

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
klasser = ["T-skjorte/topp", "Bukse", "Genser", "Kjole", "Kåpe",
           "Sandal", "Skjorte", "Joggesko", "Veske", "Ankelstøvel"]

In [ ]:
(X_full, y_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_full = (X_full / 255.0).astype("float32")[..., np.newaxis]     # (n, 28, 28, 1): én fargekanal
X_test = (X_test / 255.0).astype("float32")[..., np.newaxis]
X_train, y_train = X_full[:20_000], y_full[:20_000]
X_val, y_val = X_full[-5_000:], y_full[-5_000:]
print("Trening:", X_train.shape, " validering:", X_val.shape, " test:", X_test.shape)

## 1. En demonstrasjon av konvolusjon

Et konvolusjonslag flytter et lite filter (f.eks. 3 × 3) over bildet og regner ut en vektet sum i hver posisjon.
Resultatet kalles et *feature map*. Under bruker vi to håndlagde filtre: ett som reagerer på vertikale kanter og ett
som reagerer på horisontale kanter (jf. Géron figur 13-5).

In [ ]:
vertikal = np.array([[1, 0, -1],
                     [1, 0, -1],
                     [1, 0, -1]], dtype="float32")
horisontal = vertikal.T
filtre = np.stack([vertikal, horisontal], axis=-1)[:, :, np.newaxis, :]   # (høyde, bredde, kanaler inn, kanaler ut)

bilde = X_train[1:2]                                                        # én genser, form (1, 28, 28, 1)
fmap = tf.nn.conv2d(bilde, filtre, strides=1, padding="SAME").numpy()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(bilde[0, :, :, 0], cmap="gray_r"); ax[0].set_title(f"Input: {klasser[y_train[1]]}")
ax[1].imshow(fmap[0, :, :, 0], cmap="RdBu"); ax[1].set_title("Filter: vertikale kanter")
ax[2].imshow(fmap[0, :, :, 1], cmap="RdBu"); ax[2].set_title("Filter: horisontale kanter")
for a in ax: a.axis("off")
plt.show()

I et CNN er det ikke vi som bestemmer filtrene. Filterverdiene er vektene som læres med backpropagation.



## 2. Utdatastørrelse

For et input med bredde $n$, kjernestørrelse $f$, padding $p$ og stride $s$ blir bredden på utdata

$$n_\text{ut} = \left\lfloor \frac{n + 2p - f}{s} \right\rfloor + 1$$

`padding="valid"` betyr ingen padding ($p = 0$); `padding="same"` padder slik at $n_\text{ut} = \lceil n / s \rceil$.
Antall parametere i et `Conv2D`-lag er $(f \cdot f \cdot c_\text{inn} + 1) \cdot c_\text{ut}$ – uavhengig av bildestørrelsen.

In [ ]:
x = tf.zeros((1, 28, 28, 1))
for padding in ["valid", "same"]:
    for stride in [1, 2]:
        lag = tf.keras.layers.Conv2D(8, 3, strides=stride, padding=padding)
        print(f"padding={padding:5s} stride={stride}: utdata {tuple(lag(x).shape)}, parametere = {lag.count_params()}")

## 3. Pooling (Géron s. 363)

Et pooling-lag krymper feature maps ved å oppsummere små områder – *max pooling* tar største verdi.
Det reduserer beregningene og gjør nettverket mer robust mot små forskyvninger. Pooling har ingen vekter.

In [ ]:
liten = np.array([[1, 3, 2, 1],
                  [4, 6, 5, 0],
                  [7, 2, 9, 8],
                  [1, 0, 3, 4]], dtype="float32").reshape(1, 4, 4, 1)
print("Input:\n", liten[0, :, :, 0])
print("Max pooling 2×2:\n", tf.keras.layers.MaxPooling2D(2)(liten).numpy()[0, :, :, 0])
print("Average pooling 2×2:\n", tf.keras.layers.AveragePooling2D(2)(liten).numpy()[0, :, :, 0])

## 4. MLP vs. CNN

Et `Dense`-lag kobler hver piksel til hvert nevron og ser ikke at nabopiksler i et større bilde.
Et konvolusjonslag har lokale koblinger og delte vekter – samme filter brukes over hele bildet.

In [ ]:
def lag_mlp():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28, 1)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="MLP")

def lag_cnn():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="CNN")

mlp, cnn = lag_mlp(), lag_cnn()
cnn.summary()
print(f"Parametere – MLP: {mlp.count_params():,}   CNN: {cnn.count_params():,}")

Mønsteret i CNN-et er typisk (Géron s. 365): noen få konvolusjonslag (+ ReLU), deretter pooling, gjenta – bildet blir
mindre og antall feature maps flere – og til slutt noen fullt koblede lag og et softmax-lag.

## 5. Trening og evaluering

In [ ]:
stopp = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
historikk = {}
for m in (mlp, cnn):
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    historikk[m.name] = m.fit(X_train, y_train, epochs=10, batch_size=64,
                              validation_data=(X_val, y_val), callbacks=[stopp], verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{m.name}: testnøyaktighet = {acc:.3f}")

for navn, h in historikk.items():
    plt.plot(h.history["val_accuracy"], "o-", label=navn)
plt.xlabel("Epoke"); plt.ylabel("Valideringsnøyaktighet"); plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

y_pred = cnn.predict(X_test, verbose=0).argmax(axis=1)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=klasser, cmap="Blues",
                                        colorbar=False, xticks_rotation=60, ax=ax)
ax.grid(False)
plt.show()
print(classification_report(y_test, y_pred, target_names=klasser, digits=3))

Totalnøyaktigheten skjuler store forskjeller mellom klassene: *Skjorte* forveksles ofte med *T-skjorte*, *Genser* og *Kåpe*.
Som i samling 2: rapporter også ytelse per klasse/undergruppe.

## 6. Hva har nettverket lært?

In [ ]:
kjerner = cnn.layers[0].get_weights()[0]          # (3, 3, 1, 32)
fig, ax = plt.subplots(4, 8, figsize=(10, 5))
for i, a in enumerate(ax.ravel()):
    a.imshow(kjerner[:, :, 0, i], cmap="RdBu"); a.axis("off")
plt.suptitle("De 32 lærte 3×3-filtrene i første konvolusjonslag")
plt.show()

forste_lag = tf.keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)
fmaps = forste_lag.predict(X_test[:1], verbose=0)[0]
fig, ax = plt.subplots(4, 8, figsize=(10, 5))
for i, a in enumerate(ax.ravel()):
    a.imshow(fmaps[:, :, i], cmap="viridis"); a.axis("off")
plt.suptitle(f"Feature maps for et testbilde ({klasser[y_test[0]]})")
plt.show()

## 7. Dataaugmentering (Géron s. 309)

Med få treningsbilder kan vi lage nye, realistiske varianter ved å speile, rotere og zoome. Augmenteringslag i Keras
er bare aktive under trening, og kan legges direkte først i modellen.

In [ ]:
augmentering = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
])
fig, ax = plt.subplots(1, 8, figsize=(14, 2.2))
for a in ax:
    a.imshow(augmentering(X_train[:1], training=True)[0, :, :, 0], cmap="gray_r"); a.axis("off")
plt.suptitle("Åtte augmenterte varianter av samme bilde")
plt.show()